In [ ]:
import torch
import torchvision
import torch.nn.functional as F
import cv2
import numpy as np
import torchvision.transforms as transforms
import PIL.Image
import time
# 线程函数操作库
import threading # 线程
import ctypes
import inspect

In [ ]:
collision_model = torchvision.models.alexnet(pretrained=False)
collision_model.classifier[6] = torch.nn.Linear(collision_model.classifier[6].in_features, 2)
collision_model.load_state_dict(torch.load('best_model.pth'，map_location=torch.device(‘cuda’)))
device = torch.device('cuda')
collision_model = collision_model.to(device)

In [ ]:
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
model.load_state_dict(torch.load('best_steering_model_xy.pth'，map_location=torch.device(‘cuda’)))
device = torch.device('cuda')
model = model.to(device)
model = model.eval().half()

In [ ]:

mean = 255.0 * np.array([0.485, 0.456, 0.406])
stdev = 255.0 * np.array([0.229, 0.224, 0.225])

normalize = torchvision.transforms.Normalize(mean, stdev)

def preprocess(camera_value):
    global device, normalize
    x = camera_value
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    x = x.transpose((2, 0, 1))
    x = torch.from_numpy(x).float()
    x = normalize(x)
    x = x.to(device)
    x = x[None, ...]
    return x

In [ ]:

mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess2(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

In [ ]:
from jetbot import Robot
robot = Robot()
# 设置视觉云台的倾斜角度
robot.makerobo_servo(0,90,1,70) #根据自己的实际角度设置

In [ ]:
from IPython.display import display
import ipywidgets
import traitlets
from jetbot import Camera, bgr8_to_jpeg
import ipywidgets.widgets as widgets

In [ ]:
camera = Camera.instance(width=224, height=224)
image = widgets.Image(format='jpeg', width=224, height=224)
blocked_slider = widgets.FloatSlider(description='blocked', min=0.0, max=1.0, orientation='vertical')
image_widget = ipywidgets.Image()
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

display(widgets.HBox([image, blocked_slider]))
display(image_widget)

In [ ]:
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.0, description='steering bias')

display(speed_gain_slider, steering_gain_slider, steering_dgain_slider, steering_bias_slider)

In [ ]:
x_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='y')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='speed')

display(ipywidgets.HBox([y_slider, speed_slider]))
display(x_slider, steering_slider)

In [ ]:
#摄像头
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
face_image = widgets.Image(format='jpeg', width=640, height=480)
display(face_image)


dispW=640
dispH=480
flip=2
#Uncomment These next Two Line for Pi Camera
camSet='nvarguscamerasrc !  video/x-raw(memory:NVMM), width=3264, height=2464, format=NV12, framerate=21/1 ! nvvidconv flip-method='+str(flip)+' ! video/x-raw, width='+str(dispW)+', height='+str(dispH)+', format=BGRx ! videoconvert ! video/x-raw, format=BGR ! appsink'
cam= cv2.VideoCapture(camSet)

In [1]:
angle = 0.0
angle_last = 0.0
red_value=0   #韩明

def execute(change):
    global angle, angle_last
    image = change['new']
    xy = model(preprocess2(image)).detach().float().cpu().numpy().flatten()
    x = xy[0]
    y = (0.5 - xy[1]) / 2.0
    
    x_slider.value = x
    y_slider.value = y
    
    speed_slider.value = speed_gain_slider.value
    
    angle = np.arctan2(x, y)
    pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
    angle_last = angle
    
    steering_slider.value = pid + steering_bias_slider.value
    
    robot.left_motor.value = max(min(speed_slider.value + steering_slider.value, 1.0), 0.0)
    robot.right_motor.value = max(min(speed_slider.value - steering_slider.value, 1.0), 0.0)
    robot.makerobo_servo(0,90,1,60)
    

    
# 线程结束代码
def _async_raise(tid, exctype):
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
        raise SystemError("PyThreadState_SetAsyncExc failed")
        
def stop_thread(thread):
    _async_raise(thread.ident, SystemExit)
    
def redag():
    while True:
        ret, frame = cam.read()
        face_image.value = bgr8_to_jpeg(frame)
        if ret == False:
            break
        frame = cv2.resize(frame,(224,224))
        #截取roi区域
        roiColor = frame[0:224,0:224]
        #转换hsv颜色空间
        hsv = cv2.cvtColor(roiColor,cv2.COLOR_BGR2HSV)

        #red
        lower_hsv_red = np.array([157,177,122])
        upper_hsv_red = np.array([179,255,255])
        mask_red = cv2.inRange(hsv,lowerb=lower_hsv_red,upperb=upper_hsv_red)
        #中值滤波
        red_blur = cv2.medianBlur(mask_red, 7)
#         #green
#         lower_hsv_green = np.array([49,79,137])
#         upper_hsv_green = np.array([90,255,255])
#         mask_green = cv2.inRange(hsv,lowerb=lower_hsv_green,upperb=upper_hsv_green)
#         #中值滤波
#         green_blur = cv2.medianBlur(mask_green, 7)

        #因为图像是二值的图像，所以如果图像出现白点，也就是255，那么就取他的max最大值255
        red_color = np.max(red_blur)
#         green_color = np.max(green_blur)
        #在red_color中判断二值图像如果数值等于255，那么就判定为red
        if red_color == 255:
#             red_value=red_color  #hm
#             print(red_value)  # HM
            robot.stop()
        else :
            break
#         #在green_color中判断二值图像如果数值等于255，那么就判定为green
#         elif green_color == 255:
#             print('green')
#             break
#         c = cv2.waitKey(10)
#         if c==26:
#             break
        
    

    camera.release()
    stop_thread(t)
              

    
def update(change):
    global blocked_slider, robot
    x = change['new'] 
    x = preprocess(x)
    y = collision_model(x)
    
    # 我们应用“softmax”函数对输出向量进行标准化，使其和为1(这使其成为一个概率分布)
    y = F.softmax(y, dim=1)
    
    prob_blocked = float(y.flatten()[0])
    
    blocked_slider.value = prob_blocked
    
    if prob_blocked < 0.5:
        execute({'new': camera.value})
        camera.observe(execute, names='value')
    else:
        robot.makerobo_servo(0,90,1,110)
        t = threading.Thread(target=redag)
        t.setDaemon(True)
        t.start()
            # 更新图像小部件    
    image_widget.value = bgr8_to_jpeg(image)
    time.sleep(0.001)
update({'new': camera.value})  # 我们调用该函数一次来初始化

In [ ]:
camera.unobserve_all()
camera.observe(update, names='value')  # 这将'update'函数附加到相机的'value' traitlet上

In [ ]:
import time

camera.unobserve_all()
time.sleep(1.0)
robot.stop(90,70)